# 2 — Snowpark
## "Python DataFrames at Snowflake scale"

Snowpark is a DataFrame API that looks like pandas and executes like a database.

The critical difference: a Snowpark DataFrame is **lazy**. It holds a query, not data. Nothing moves until you call `.show()`, `.to_pandas()`, or `.collect()` — and when you do, only the result comes back.

That single property is what removes the memory ceiling.

In [ ]:
USE WAREHOUSE GUARDANT_DEMO_WH;
USE SCHEMA DEMO.GUARDANT_DEMO;

### The session

Inside a Snowflake Notebook there is nothing to authenticate. `get_active_session()` hands you the session the notebook is already running in.

In [ ]:
from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F
from snowflake.snowpark.window import Window

session = get_active_session()

variants  = session.table("VARIANT_CALLS")
specimens = session.table("SPECIMENS")
patients  = session.table("PATIENTS")
panel     = session.table("GENE_PANEL")

# Project the panel with a distinct join key. Joining two DataFrames that both
# carry GENE_SYMBOL leaves that name ambiguous downstream, so rename it once
# here rather than disambiguating at every later reference.
panel_lookup = panel.select(
    F.col("GENE_SYMBOL").alias("PANEL_GENE"),
    "IS_ACTIONABLE",
    "TARGETED_THERAPY",
)

print(f"VARIANT_CALLS rows: {variants.count():,}")

### Proof that it is lazy

Build a multi-stage pipeline — filter, join, aggregate, window — and then look at what Snowpark actually produced. It is SQL. No rows have moved.

In [ ]:
pipeline = (
    variants
    .filter(F.col("CALL_FILTER") == "PASS")
    .join(specimens, on="SPECIMEN_ID")
    .filter(F.col("QC_STATUS") == "PASS")
    .group_by("GENE_SYMBOL", "ASSAY")
    .agg(F.count("*").alias("N_CALLS"))
)

# The DataFrame is a query. Here it is.
print(pipeline.queries["queries"][0][:900])

### An aggregation across 20M rows

Per-gene statistics over the full raw table, no pre-filtering. On a laptop this requires the extract first. Here it is one expression, and the result is small enough to hand straight to pandas.

In [ ]:
import time

gene_stats = (
    variants
    .join(panel_lookup, variants["GENE_SYMBOL"] == panel_lookup["PANEL_GENE"])
    .group_by("GENE_SYMBOL", "IS_ACTIONABLE")
    .agg(
        F.count("*").alias("TOTAL_CALLS"),
        F.sum(F.iff(F.col("CALL_FILTER") == "PASS", 1, 0)).alias("PASS_CALLS"),
        F.count_distinct("SPECIMEN_ID").alias("SPECIMENS"),
        F.round(F.median("VAF") * 100, 3).alias("MEDIAN_VAF_PCT"),
        F.round(F.max("VAF") * 100, 2).alias("MAX_VAF_PCT"),
        F.round(F.avg("READ_DEPTH")).alias("MEAN_DEPTH"),
    )
    .sort(F.col("TOTAL_CALLS").desc())
)

t0 = time.time()
result = gene_stats.to_pandas()
elapsed = time.time() - t0

print(f"Aggregated 20,000,000 rows in {elapsed:.1f}s")
print(f"Returned {len(result)} rows to Python\n")
result.head(12)

### Window functions: longitudinal VAF tracking

The clinically interesting question is not "what is the VAF" — it is "is the VAF rising across serial draws". That is a window function over each patient's draw history.

This is precisely the kind of operation that gets painful in pandas at scale, because it needs the whole partition in memory at once. Snowflake does it in the warehouse.

In [ ]:
# Per patient / gene, track the clonal variant across successive draws.
clonal = (
    variants
    .filter((F.col("CALL_FILTER") == "PASS") & (F.col("VAF") >= 0.02))
    .join(specimens, on="SPECIMEN_ID")
    .filter(F.col("QC_STATUS") == "PASS")
    .select("PATIENT_ID", "SPECIMEN_ID", "GENE_SYMBOL",
            "COLLECTION_DATE", "DRAW_NUMBER", "VAF", "TUMOR_FRACTION")
)

w = Window.partition_by("PATIENT_ID", "GENE_SYMBOL").order_by("COLLECTION_DATE")

trajectory = (
    clonal
    .with_column("PREV_VAF", F.lag("VAF").over(w))
    .with_column("PREV_DATE", F.lag("COLLECTION_DATE").over(w))
    .filter(F.col("PREV_VAF").is_not_null())
    .with_column("VAF_DELTA", F.round((F.col("VAF") - F.col("PREV_VAF")) * 100, 3))
    .with_column("DAYS_BETWEEN", F.datediff("day", F.col("PREV_DATE"), F.col("COLLECTION_DATE")))
    .filter(F.col("DAYS_BETWEEN") > 0)
    .with_column("VAF_TREND", F.iff(F.col("VAF_DELTA") > 0, F.lit("rising"), F.lit("falling")))
)

rising = (
    trajectory
    .group_by("GENE_SYMBOL")
    .agg(
        F.count("*").alias("PAIRED_OBSERVATIONS"),
        F.sum(F.iff(F.col("VAF_DELTA") > 0, 1, 0)).alias("RISING"),
        F.round(F.avg("VAF_DELTA"), 4).alias("MEAN_VAF_DELTA_PCT"),
    )
    .with_column("PCT_RISING", F.round(100 * F.col("RISING") / F.col("PAIRED_OBSERVATIONS"), 1))
    .sort(F.col("PAIRED_OBSERVATIONS").desc())
)

rising.to_pandas().head(12)

### Reusable logic: a UDF

Right now a scoring rule like this probably lives in a Python file on someone's machine, and drifts between team members. Registered as a UDF it runs inside Snowflake, is versioned as an object, and is callable from SQL by anyone with the grant — including analysts who do not write Python.

In [ ]:
from snowflake.snowpark.types import FloatType, StringType, IntegerType, BooleanType

def confidence_score(vaf: float, depth: int, alt_reads: int, mapq: int, actionable: bool) -> float:
    """Composite confidence that a call is a true somatic variant worth reporting."""
    if vaf is None or depth is None or alt_reads is None or mapq is None:
        return 0.0
    score = 0.0
    score += min(vaf / 0.10, 1.0) * 35          # allele fraction
    score += min(depth / 5000.0, 1.0) * 25      # coverage
    score += min(alt_reads / 50.0, 1.0) * 20    # supporting reads
    score += min(max(mapq - 20, 0) / 40.0, 1.0) * 15   # mapping quality
    if actionable:
        score += 5                              # clinical relevance nudge
    return round(min(score, 100.0), 2)

session.udf.register(
    func=confidence_score,
    name="VARIANT_CONFIDENCE_SCORE",
    return_type=FloatType(),
    input_types=[FloatType(), IntegerType(), IntegerType(), IntegerType(), BooleanType()],
    is_permanent=True,
    stage_location="@DEMO_STAGE",
    replace=True,
)

print("Registered DEMO.GUARDANT_DEMO.VARIANT_CONFIDENCE_SCORE")

The same function, now callable from SQL. This is the handoff moment: the data scientist writes it in Python, the analyst uses it in SQL.

In [ ]:
SELECT specimen_id,
       gene_symbol,
       ROUND(vaf * 100, 3)                AS vaf_pct,
       read_depth,
       alt_read_count,
       clinical_significance,
       VARIANT_CONFIDENCE_SCORE(vaf, read_depth, alt_read_count,
                                mapping_quality, is_actionable) AS confidence
FROM V_REPORTABLE_VARIANTS
WHERE is_actionable
QUALIFY ROW_NUMBER() OVER (ORDER BY confidence DESC) <= 20
ORDER BY confidence DESC;

### Writing results back

`save_as_table` persists a Snowpark DataFrame with no extract-transform-upload round trip. The cohort table below is built entirely server-side.

In [ ]:
cohort = (
    variants
    .filter(F.col("CALL_FILTER") == "PASS")
    .join(panel_lookup, variants["GENE_SYMBOL"] == panel_lookup["PANEL_GENE"])
    .filter(F.col("IS_ACTIONABLE") & (F.col("VAF") >= 0.05))
    .join(specimens, on="SPECIMEN_ID")
    .filter(F.col("QC_STATUS") == "PASS")
    .join(patients, on="PATIENT_ID")
    .select(
        "PATIENT_ID", "SPECIMEN_ID", "PRIMARY_CANCER_TYPE", "STAGE_AT_DIAGNOSIS",
        "COLLECTION_DATE", "ASSAY", "TUMOR_FRACTION",
        "GENE_SYMBOL", "CONSEQUENCE", "VAF", "READ_DEPTH",
        "CLINICAL_SIGNIFICANCE", "TARGETED_THERAPY",
    )
)

cohort.write.mode("overwrite").save_as_table("ACTIONABLE_COHORT")

print(f"ACTIONABLE_COHORT written: {session.table('ACTIONABLE_COHORT').count():,} rows")

---

### What changed

| | pandas on a laptop | Snowpark |
|---|---|---|
| Where it runs | Local CPU / RAM | Snowflake warehouse |
| 20M-row aggregate | Extract first, then hope | One expression |
| Scaling up | Buy a bigger laptop | Resize the warehouse |
| Sharing logic | Copy the .py file around | Register a UDF, grant it |
| Writing back | Upload step | `save_as_table` |

Next: **notebook 3** — the same platform, applied to the unstructured half of the data.